In [3]:
pip install yfinance

  Using cached yfinance-1.0-py2.py3-none-any.whl.metadata (6.0 kB)
  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata (13 kB)
Using cached yfinance-1.0-py2.py3-none-any.whl (127 kB)
Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl (1.6 MB)

   ---------------------------------------- 0/5 [peewee]
   ---------------------------------------- 0/5 [peewee]
   ---------------------------------------- 0/5 [peewee]
   ---------------- ----------------------- 2/5 [websockets]
   ---------------- ----------------------- 2/5 [websockets]
   ---------------- ----------------------- 2/5 [websockets]
   ---------------- ----------------------- 2/5 [websockets]
   ------------------------ --------------- 3/5 [curl_cffi]
   ------------------------ --------------- 3/5 [curl_cffi]
   -------------------------------- ------- 4/5 [yfinance]
   -------------------------------- ------- 4/5 [yfinance]
   --------------------------------

In [4]:
import psycopg2
import pandas as pd
import yfinance as yf

# Create the 'visa' Database

# Connect to the default 'postgres' database first
try: 
    conn = psycopg2.connect("host=127.0.0.1 dbname=postgres user=postgres password=Shifa.326459")
except psycopg2.Error as e: 
    print("Error: Could not make connection to the Postgres database")
    print(e)

try: 
    cur = conn.cursor()
except psycopg2.Error as e: 
    print("Error: Could not get curser to the Database")
    print(e)

# Set autocommit to True to allow CREATE DATABASE command
conn.set_session(autocommit=True)

In [5]:
# Create the database (Drop if exists to start fresh)
try:
    cur.execute("DROP DATABASE IF EXISTS visa")
    cur.execute("CREATE DATABASE visa")
    print("Database 'visa' created successfully.")
except psycopg2.Error as e:
    print("Error: Could not create database")
    print(e)

# Close the connection to 'postgres'
conn.close()

Database 'visa' created successfully.


In [6]:
# Create Table and Insert Data

# Connect to the new 'visa' database
try: 
    conn = psycopg2.connect("host=127.0.0.1 dbname=visa user=postgres password=Shifa.326459")
except psycopg2.Error as e: 
    print("Error: Could not make connection to the new database")
    print(e)

try: 
    cur = conn.cursor()
except psycopg2.Error as e: 
    print("Error: Could not get curser to the Database")
    print(e)

conn.set_session(autocommit=True)

In [7]:
# Create the table structure
try:
    cur.execute("""
        CREATE TABLE IF NOT EXISTS stock_data (
            transaction_date DATE PRIMARY KEY,
            open_price NUMERIC,
            high_price NUMERIC,
            low_price NUMERIC,
            close_price NUMERIC,
            volume BIGINT
        );
    """)
    print("Table 'stock_data' created successfully.")
except psycopg2.Error as e:
    print("Error: Could not create table")
    print(e)

Table 'stock_data' created successfully.


In [8]:
# Download Visa (V) stock data
print("Downloading data for Visa (V)...")
ticker = yf.Ticker("V")
df = ticker.history(period="5y")

# Reset index to make Date a column
df.reset_index(inplace=True)

In [9]:
# Insert data into the database
try:
    for index, row in df.iterrows():
        sql_insert = """
            INSERT INTO stock_data (transaction_date, open_price, high_price, low_price, close_price, volume)
            VALUES (%s, %s, %s, %s, %s, %s)
        """
        # Format the date to ensure it matches SQL format
        date_val = row['Date'].date()
        
        cur.execute(sql_insert, (
            date_val, 
            row['Open'], 
            row['High'], 
            row['Low'], 
            row['Close'], 
            row['Volume']
        ))
    
    print("Data insertion completed successfully.")

except psycopg2.Error as e:
    print("Error: Could not insert data")
    print(e)

Data insertion completed successfully.


In [10]:
# Retrieve and Display Data from 'visa' Database

try:
    # 2. Create a SQL query to select the latest 10 records
    sql_query = """
        SELECT * FROM stock_data 
        ORDER BY transaction_date DESC 
        LIMIT 10;
    """
    
    # 3. Use Pandas to read the SQL result into a DataFrame
    df = pd.read_sql_query(sql_query, conn)
    
    # 4. Display the data
    print("--- Latest 10 records from Visa stock data ---")
    print(df)

except psycopg2.Error as e:
    print("Error: Could not fetch data from the database")
    print(e)

--- Latest 10 records from Visa stock data ---
  transaction_date  open_price  high_price   low_price  close_price    volume
0       2026-01-14  328.660004  329.899994  323.940002   329.170013   9385800
1       2026-01-13  337.000000  337.519989  323.829987   327.880005  20383500
2       2026-01-12  342.779999  346.510010  337.320007   343.200012  13281300
3       2026-01-09  352.160004  354.700012  349.160004   349.769989   4902900
4       2026-01-08  355.000000  356.350006  349.500000   352.230011   6357600
5       2026-01-07  357.149994  358.279999  354.510010   355.880005   6333900
6       2026-01-06  353.679993  358.619995  352.000000   357.559998   6775400
7       2026-01-05  344.500000  357.540009  344.049988   353.799988   7592200
8       2026-01-02  349.869995  350.049988  343.480011   346.480011   5403800
9       2025-12-31  353.649994  355.200012  350.690002   350.709991   3503200


C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_20952\4267683213.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(sql_query, conn)


In [11]:
import psycopg2
import pandas as pd

try:   
    # 2. Fetch statistical data from the database
    # COUNT(*) = Count total rows
    # MIN(...) = Find earliest date
    # MAX(...) = Find latest date
    sql_audit = """
        SELECT 
            COUNT(*) as total_rows,
            MIN(transaction_date) as first_date,
            MAX(transaction_date) as last_date
        FROM stock_data;
    """
    
    df_audit = pd.read_sql_query(sql_audit, conn)
    
    # 3. Display audit results
    print("===== Data Audit Report =====")
    print(f"Total Rows: {df_audit.iloc[0]['total_rows']} records")
    print(f"Start Date: {df_audit.iloc[0]['first_date']}")
    print(f"End Date  : {df_audit.iloc[0]['last_date']}")
    print("=============================")

    # 4. Check for zero or negative prices (Data Quality Check)
    sql_check_zero = "SELECT * FROM stock_data WHERE close_price <= 0;"
    df_zero = pd.read_sql_query(sql_check_zero, conn)
    
    if len(df_zero) == 0:
        print("Success: No zero or negative prices found. Data is clean.")
    else:
        print(f"Warning: Found {len(df_zero)} rows with zero or negative prices.")

except Exception as e:
    print("Error:", e)

===== Data Audit Report =====
Total Rows: 1255 records
Start Date: 2021-01-15
End Date  : 2026-01-14
Success: No zero or negative prices found. Data is clean.


C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_20952\940261686.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_audit = pd.read_sql_query(sql_audit, conn)
C:\Users\Zbook Firefly 14 G8\AppData\Local\Temp\ipykernel_20952\940261686.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_zero = pd.read_sql_query(sql_check_zero, conn)
